In [27]:
import pandas as pd
import numpy as np
import joblib

# Load artifacts
model = joblib.load("health_model.pkl")
scaler = joblib.load("scaler.pkl")

# الأعمدة الرقمية اللي اتعملها scaling أثناء التدريب
numeric_cols = scaler.feature_names_in_

# =========================
# Test cases (profiles)
# =========================
test_data = pd.DataFrame([
    # 🟢 Case 1: Very healthy
    {
        "Age": 28,
        "BMI": 22,
        "Exercise_Frequency": 5,
        "Diet_Quality": 90,
        "Sleep_Hours": 8,
        "Alcohol_Consumption": 1,
        "Smoking_Status": 0
    },

    # 🟡 Case 2: Average
    {
        "Age": 40,
        "BMI": 27,
        "Exercise_Frequency": 2,
        "Diet_Quality": 65,
        "Sleep_Hours": 6.5,
        "Alcohol_Consumption": 4,
        "Smoking_Status": 0
    },

    # 🔴 Case 3: Unhealthy
    {
        "Age": 50,
        "BMI": 33,
        "Exercise_Frequency": 0,
        "Diet_Quality": 45,
        "Sleep_Hours": 5,
        "Alcohol_Consumption": 8,
        "Smoking_Status": 1
    },

    # 🔵 Case 4: Smoker but otherwise healthy
    {
        "Age": 35,
        "BMI": 24,
        "Exercise_Frequency": 4,
        "Diet_Quality": 80,
        "Sleep_Hours": 7,
        "Alcohol_Consumption": 2,
        "Smoking_Status": 1
    },

    # 🟣 Case 5: High diet quality compensates
    {
        "Age": 45,
        "BMI": 30,
        "Exercise_Frequency": 1,
        "Diet_Quality": 85,
        "Sleep_Hours": 7,
        "Alcohol_Consumption": 3,
        "Smoking_Status": 0
    }
])

# =========================
# Preprocessing
# =========================

# Log transform (زي training)
test_data["Alcohol_Consumption"] = np.log1p(
    test_data["Alcohol_Consumption"]
)

# Scaling (من غير Smoking_Status)
test_data[numeric_cols] = scaler.transform(
    test_data[numeric_cols]
)

# ترتيب الأعمدة زي ما الموديل اتدرّب
test_data = test_data[model.feature_names_in_]

# =========================
# Prediction
# =========================
preds = model.predict(test_data)
preds = np.clip(preds, 0, 100)

# =========================
# Results
# =========================
results = test_data.copy()
results["Predicted_Health_Score"] = preds

print(results[["Predicted_Health_Score"]])


   Predicted_Health_Score
0              100.000000
1               79.042419
2               26.458950
3               96.312481
4               88.180554
